# Entropy-Augmented Reward for SRT: Fixing the Reward Hack

**Project:** Extending Self-Rewarding Training (SRT) with a batch-entropy bonus to prevent reward collapse.  
**Dataset:** [DAPO](https://huggingface.co/datasets/ftajwar/deduplicated_dapo_dataset) (`ftajwar/deduplicated_dapo_dataset`)  

**Models** — all open-weight, downloaded from HuggingFace Hub, **no API key required**:

| Model | HF Hub ID | Why |
|---|---|---|
| Qwen2.5-Math-7B | `Qwen/Qwen2.5-Math-7B` | Native math pre-training; project baseline in `srt.sh` |
| DeepSeek-R1-Distill-Qwen-7B | `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` | RL-distilled CoT reasoning |
| NuminaMath-7B-CoT | `AI-MO/NuminaMath-7B-CoT` | Competition-math fine-tuned |

**How models are loaded** — identical to the verl/vLLM pipeline:
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
```
Weights are cached under `~/.cache/huggingface/hub` after the first download.

**The problem:** In standard SRT the self-consistency pseudo-reward $R_{consistency}$ is gamed when the model collapses to always emitting the same answer. Every generation matches — pseudo-reward → 1.0 — but true accuracy → 0.

**The fix:**
$$R_{total} = R_{consistency} + \alpha \, \mathcal{H}_{batch}$$
where $\mathcal{H}_{batch}$ is the Shannon entropy of the answer distribution in the current batch.

---
## Step 0 — Imports & Config

In [1]:
import sys, os
# Make sure verl is importable from the repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) != "srtorigin" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple

import torch
import datasets
from transformers import AutoTokenizer, AutoModelForCausalLM

import matplotlib.pyplot as plt

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Choose which model to run in Steps 1-4 ───────────────────────────
# Switch between the three by changing this line.
MODEL_ID = "Qwen/Qwen2.5-Math-7B"                      # baseline model (matches srt.sh)
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # alternative
# MODEL_ID = "AI-MO/NuminaMath-7B-CoT"                  # alternative

DATA_SOURCE   = "medqa"       # MedQA-USMLE 4-option multiple-choice
LETTERS       = ["A", "B", "C", "D"]
N_TEST        = 100           # fixed test set size
N_GENERATIONS = 16            # rollouts per prompt (matches srt.sh NUM_PER_PROMPT_ROLLOUTS)
MAX_NEW_TOKENS = 32           # minimum for a single \boxed{letter} response
TEMPERATURE    = 1.0
TOP_P          = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Model : {MODEL_ID}")


ModuleNotFoundError: No module named 'torch'

---
## Step 1 — Environment & Baseline Verification (The 100-Sample Test)

Before introducing the fix we must reproduce the failure.

### 1a — Load DAPO and build the 100-sample test set

In [4]:
def format_mc_prompt(passage: str, question: str, options: List[str]) -> str:
    """Format a multiple-choice question asking for \\boxed{letter}."""
    parts = []
    if passage and passage.strip():
        parts.append(f"Passage: {passage.strip()}\n")
    parts.append(f"Question: {question.strip()}\n")
    for letter, opt in zip(LETTERS, options):
        parts.append(f"{letter}) {opt}")
    parts.append("\nSelect the correct answer. Output only the letter within \\boxed{}.")
    return "\n".join(parts)


def build_test_set(split, n: int, seed: int = SEED) -> List[Dict]:
    """Extract n random rows from MedQA-USMLE-4-options and format as MC prompts.

    Schema: question (str), options (dict {A:..., B:..., C:..., D:...}),
            answer (str, full text), answer_idx (str, letter A-D).
    """
    rng     = random.Random(seed)
    indices = rng.sample(range(len(split)), n)
    rows    = []
    for idx in indices:
        ex = split[idx]
        passage  = ex.get("passage", ex.get("context", ex.get("text", "")))
        question = ex.get("question", ex.get("query", ""))

        # options can be a dict {A:..., B:..., C:..., D:...} (MedQA)
        # or a list (LogiQA-style) — handle both
        raw_opts = ex.get("options", ex.get("choices", []))
        if isinstance(raw_opts, dict):
            options = [raw_opts[k] for k in sorted(raw_opts.keys())]
        else:
            options = list(raw_opts)

        # answer_idx is a letter (A-D) in MedQA; fallback to integer index
        answer = ex.get("answer_idx", ex.get("answer", ex.get("label", 0)))
        if isinstance(answer, str) and answer.upper() in LETTERS:
            correct_idx = LETTERS.index(answer.upper())
        else:
            try:
                correct_idx = int(answer)
            except (ValueError, TypeError):
                correct_idx = 0

        correct_letter = LETTERS[correct_idx]
        rows.append({
            "original_idx":  idx,
            "data_source":   DATA_SOURCE,
            "passage":       passage,
            "question":      question,
            "options":       options,
            "correct_idx":   correct_idx,
            "hidden_answer": correct_letter,
            "ground_truth":  SELF_LABEL_SENTINEL,
            "prompt":        format_mc_prompt(passage, question, options),
        })
    return rows


test_set = build_test_set(train_split, N_TEST)
df_test  = pd.DataFrame([{k: v for k, v in r.items() if k != "options"} for r in test_set])
print(f"Test set: {len(df_test)} prompts")
df_test.head(3)


Test set: 100 prompts


,original_idx,data_source,passage,question,correct_idx,hidden_answer,ground_truth,prompt
0,912,medqa,,A 55-year-old woman presents to the office com...,1,B,LABEL_BY_SELF_CONSISTENCY,Question: A 55-year-old woman presents to the ...
1,204,medqa,,A 59-year-old woman comes to the physician bec...,2,C,LABEL_BY_SELF_CONSISTENCY,Question: A 59-year-old woman comes to the phy...
2,2253,medqa,,A 12-year-old boy presents with progressive cl...,2,C,LABEL_BY_SELF_CONSISTENCY,Question: A 12-year-old boy presents with prog...


### 1b — Load the model (same as verl does internally)

In [5]:
# verl loads models with from_pretrained + bfloat16 + device_map="auto".
# This cell does the same.  Weights are downloaded once and cached locally.
# No API key is needed — these are all public HuggingFace Hub models.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",       # spreads across available GPUs automatically
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded: {MODEL_ID}")
print(f"Parameters : {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 331.15it/s]
Some parameters are on the meta device because they were offloaded to the disk.


Model loaded: Qwen/Qwen2.5-Math-7B
Parameters : 7.6B


### 1c — Scoring utilities (imported directly from verl)

We reuse the **exact same** `_default_compute_score` and `_extract_verifiable_part_of_solution` functions
that `SelfLearningRewardManager` uses in `verl/workers/reward_manager/self_learning.py`.

In [6]:
def extract_answer(text: str, data_source: str = DATA_SOURCE) -> Optional[str]:
    """Extract A/B/C/D from model output using several fallback patterns."""
    # Pattern 1: \boxed{B} or \\boxed{B}
    m = re.search(r'\\{1,2}boxed\{([A-D])\}', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # Pattern 2: "the answer is B" / "answer: B"
    m = re.search(r'(?:the\s+answer\s+is|answer\s*:)\s*([A-D])\b', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # Pattern 3: last standalone A/B/C/D letter in the text
    hits = re.findall(r'\b([A-D])\b', text.upper())
    return hits[-1] if hits else None


def compute_score(response: str, ground_truth: str, data_source: str = DATA_SOURCE) -> float:
    """1.0 if extracted letter matches ground_truth, else 0.0."""
    extracted = extract_answer(response)
    if extracted is None:
        return 0.0
    return 1.0 if extracted == ground_truth.strip().upper() else 0.0


# Sanity checks
assert extract_answer(r"The answer is \boxed{B}.")       == "B", "boxed pattern failed"
assert extract_answer("Therefore the answer is C")        == "C", "answer-is pattern failed"
assert extract_answer("I believe D is correct")           == "D", "last-letter pattern failed"
assert compute_score(r"The answer is \boxed{A}", "A")    == 1.0, "correct score wrong"
assert compute_score(r"The answer is \boxed{A}", "B")    == 0.0, "wrong score should be 0"
print("MC scoring utilities: OK")


MC scoring utilities: OK


### 1d — Generation helper (mirrors verl rollout worker)

In [ ]:
@torch.inference_mode()
def generate_rollouts(
    prompt: str,
    n: int = N_GENERATIONS,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
) -> List[str]:
    """Generate n independent rollouts for a single (fixed) prompt."""
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        formatted = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        formatted = prompt

    inputs = tokenizer(
        [formatted] * n,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    return [
        tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        for out in outputs
    ]


def generate_shuffled_rollouts(
    row: Dict,
    n: int = N_GENERATIONS,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
) -> Tuple[List[str], List[str]]:
    """
    Generate n rollouts for one question, each rollout seeing a DIFFERENT
    random ordering of the 4 answer options.

    Returns
    -------
    responses       : n raw model output strings
    correct_letters : n correct letters (one per rollout, reflecting the shuffle)

    Why shuffle?
      - Eliminates position bias (model always picking "A").
      - Forces genuine reasoning instead of memorising answer position.
      - Creates natural diversity in the batch for the entropy signal.
    """
    responses, correct_letters = [], []
    for _ in range(n):
        perm            = list(range(len(row["options"])))
        random.shuffle(perm)
        shuffled_opts   = [row["options"][i] for i in perm]
        new_correct_pos = perm.index(row["correct_idx"])
        correct_letters.append(LETTERS[new_correct_pos])

        prompt   = format_mc_prompt(row["passage"], row["question"], shuffled_opts)
        response = generate_rollouts(
            prompt, n=1, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=top_p
        )[0]
        responses.append(response)
    return responses, correct_letters


# ── Smoke test: 2 shuffled rollouts on the first LogiQA question ─────
print("=== Smoke test: LogiQA 2.0 + per-rollout shuffled options ===\n")
_row = test_set[0]
print(f"Question : {_row['question'][:120]}")
print(f"Options  : {_row['options']}")
print(f"Correct  : {_row['hidden_answer']}\n")

_responses, _correct_letters = generate_shuffled_rollouts(_row, n=2, max_new_tokens=64)
for i, (resp, cl) in enumerate(zip(_responses, _correct_letters)):
    extracted = extract_answer(resp)
    score     = compute_score(resp, cl)
    print(f"--- Rollout {i+1}  (correct answer for this shuffle: {cl}) ---")
    print(f"Response  : {resp[:250]}")
    print(f"Extracted : {extracted}   Score: {score}\n")


### 1d-ii — Reward function test (1/0) with and without rollouts

Tests the original SRT reward signal on a single question: score one response directly, then score each shuffled rollout independently.

In [ ]:
# ── Reward function test: 1/0 scores with and without rollouts ────────
# This mirrors the original SRT reward logic from the paper:
#   - WITHOUT rollouts: score a single response directly against ground truth
#   - WITH rollouts: score each rollout, see how many are correct (1) or wrong (0)

print("=== Original Reward Function Test (1 / 0) ===\n")

_row = test_set[0]
print(f"Question : {_row['question']}")
print(f"Options  : {_row['options']}")
print(f"Correct  : {_row['hidden_answer']}\n")

# ── WITHOUT rollout: single response ─────────────────────────────────
print("--- WITHOUT rollouts (single generation) ---")
single_prompt   = format_mc_prompt(_row["passage"], _row["question"], _row["options"])
single_response = generate_rollouts(single_prompt, n=1, max_new_tokens=64)[0]
single_score    = compute_score(single_response, _row["hidden_answer"])

print(f"Response  : {single_response[:300]}")
print(f"Extracted : {extract_answer(single_response)}")
print(f"Score     : {int(single_score)}  (1=correct, 0=wrong)\n")

# ── WITH rollouts: 4 shuffled responses ──────────────────────────────
print("--- WITH rollouts (4 shuffled generations) ---")
_resps, _cls = generate_shuffled_rollouts(_row, n=4, max_new_tokens=64)

scores = [compute_score(r, cl) for r, cl in zip(_resps, _cls)]
print(f"{'Rollout':<10} {'Shuffle correct':<18} {'Extracted':<12} {'Score'}")
print("-" * 52)
for i, (r, cl, s) in enumerate(zip(_resps, _cls, scores)):
    ext = extract_answer(r) or "None"
    print(f"  {i+1:<8} {cl:<18} {ext:<12} {int(s)}")

print(f"\nPass rate: {sum(scores)}/{len(scores)} = {sum(scores)/len(scores):.2f}")
print(f"Majority answer: {Counter(extract_answer(r) for r in _resps if extract_answer(r)).most_common(1)}")


### 1d-iii — Reward function comparison: Old vs Old+Shuffle vs New

Runs the **same question** through all three reward setups so you can compare them directly:

| | Option order | Reward formula |
|---|---|---|
| **Old** | Fixed (same every rollout) | `R = 1 or 0` |
| **Old + Shuffle** | Different per rollout | `R = 1 or 0` |
| **New** | Different per rollout | `R = (1 or 0) + α·H` |

In [ ]:
# ── Side-by-side comparison of all three reward approaches ───────────
# Uses the same question and same model — only the reward function changes.
#
#  OLD          : fixed option order, consistency reward only (R = 0 or 1)
#  OLD + SHUFFLE: shuffled option order per rollout, consistency reward only
#  NEW          : shuffled order + entropy bonus  (R = consistency + α·H)

N_DEMO  = 4      # rollouts to show (keep small so it runs fast)
ALPHA   = 0.1    # entropy weight for the NEW reward
_row    = test_set[0]

print("=" * 65)
print(f"Question : {_row['question'][:100]}")
print(f"Options  : {_row['options']}")
print(f"Correct  : {_row['hidden_answer']}")
print("=" * 65)

# ── 1. OLD: same fixed prompt for every rollout ───────────────────────
print("\n[1] OLD reward  (fixed order, consistency only)\n")
fixed_prompt    = format_mc_prompt(_row["passage"], _row["question"], _row["options"])
fixed_responses = generate_rollouts(fixed_prompt, n=N_DEMO, max_new_tokens=64)
fixed_scores    = [compute_score(r, _row["hidden_answer"]) for r in fixed_responses]

for i, (r, s) in enumerate(zip(fixed_responses, fixed_scores)):
    print(f"  Rollout {i+1}: extracted={extract_answer(r) or 'None':>4}  score={int(s)}")
print(f"  Pass rate : {sum(fixed_scores)}/{N_DEMO} = {sum(fixed_scores)/N_DEMO:.2f}")
print(f"  Pseudo-reward (majority-vote consistency): "
      f"{Counter(extract_answer(r) for r in fixed_responses if extract_answer(r)).most_common(1)}")

# ── 2. OLD + SHUFFLE: different option order per rollout ──────────────
print("\n[2] OLD + SHUFFLE  (shuffled order, consistency only)\n")
shuf_responses, shuf_correct = generate_shuffled_rollouts(_row, n=N_DEMO, max_new_tokens=64)
shuf_scores = [compute_score(r, cl) for r, cl in zip(shuf_responses, shuf_correct)]

for i, (r, cl, s) in enumerate(zip(shuf_responses, shuf_correct, shuf_scores)):
    print(f"  Rollout {i+1}: correct_this_shuffle={cl}  extracted={extract_answer(r) or 'None':>4}  score={int(s)}")
print(f"  Pass rate : {sum(shuf_scores)}/{N_DEMO} = {sum(shuf_scores)/N_DEMO:.2f}")

# ── 3. NEW: shuffled order + entropy bonus ────────────────────────────
print(f"\n[3] NEW reward  (shuffled order + entropy bonus, α={ALPHA})\n")
extracted_shuf = [extract_answer(r) for r in shuf_responses]
H              = compute_batch_entropy(extracted_shuf, N_DEMO)
entropy_bonus  = ALPHA * H

for i, (r, cl, s) in enumerate(zip(shuf_responses, shuf_correct, shuf_scores)):
    total = s + entropy_bonus
    print(f"  Rollout {i+1}: consistency={int(s)}  entropy_bonus={entropy_bonus:.3f}  "
          f"R_total={total:.3f}")

print(f"\n  Batch entropy H        : {H:.4f}  (0=all same answer, 1=all different)")
print(f"  Entropy bonus (α·H)    : {entropy_bonus:.4f}")
print(f"  Mean consistency reward: {sum(shuf_scores)/N_DEMO:.4f}")
print(f"  Mean total reward      : {(sum(shuf_scores)/N_DEMO) + entropy_bonus:.4f}")

# ── Summary table ─────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"{'Approach':<30} {'Pass rate':>10} {'Mean reward':>12}")
print("-" * 65)
print(f"{'[1] Old (fixed order)':<30} {sum(fixed_scores)/N_DEMO:>10.2f} {sum(fixed_scores)/N_DEMO:>12.4f}")
print(f"{'[2] Old + shuffle':<30} {sum(shuf_scores)/N_DEMO:>10.2f} {sum(shuf_scores)/N_DEMO:>12.4f}")
print(f"{'[3] New (shuffle + entropy)':<30} {sum(shuf_scores)/N_DEMO:>10.2f} {(sum(shuf_scores)/N_DEMO)+entropy_bonus:>12.4f}")
print("=" * 65)


### 1e — Run baseline on all 100 samples (16 generations each)

This replicates what verl does each training step: generate `NUM_PER_PROMPT_ROLLOUTS=16` responses per prompt and score them.

In [ ]:
def run_baseline_rollouts(
    prompts: List[Dict],
    n_gen: int = N_GENERATIONS,
) -> List[Dict]:
    """
    For each prompt: generate n_gen shuffled rollouts and score them.
    Each rollout sees a different option ordering (see generate_shuffled_rollouts).
    This mirrors the verl rollout worker from the original SRT codebase.
    """
    results = []
    for i, row in enumerate(prompts):
        responses, correct_letters = generate_shuffled_rollouts(row, n=n_gen)
        extracted = [extract_answer(r) for r in responses]
        scores    = [compute_score(r, cl) for r, cl in zip(responses, correct_letters)]

        valid        = [a for a in extracted if a is not None]
        majority_ans = Counter(valid).most_common(1)[0][0] if valid else None
        pseudo_r     = Counter(valid).most_common(1)[0][1] / len(valid) if valid else 0.0

        results.append({
            "prompt":          row["prompt"],
            "hidden_answer":   row["hidden_answer"],
            "responses":       responses,
            "correct_letters": correct_letters,
            "extracted":       extracted,
            "true_scores":     scores,
            "true_accuracy":   float(np.mean(scores)),
            "majority_ans":    majority_ans,
            "pseudo_reward":   pseudo_r,
        })

        if (i + 1) % 10 == 0:
            avg_acc = np.mean([r["true_accuracy"] for r in results])
            avg_pr  = np.mean([r["pseudo_reward"]  for r in results])
            print(f"[{i+1:3d}/{len(prompts)}]  avg_true_acc={avg_acc:.3f}  avg_pseudo_reward={avg_pr:.3f}")

    return results


print("Running baseline rollouts on 100 LogiQA 2.0 samples...")
baseline_rollouts = run_baseline_rollouts(test_set)

df_baseline = pd.DataFrame([
    {"prompt": r["prompt"],
     "true_accuracy": r["true_accuracy"],
     "pseudo_reward": r["pseudo_reward"]}
    for r in baseline_rollouts
])
print(f"\nBaseline summary (n=100 prompts, {N_GENERATIONS} shuffled rollouts each):")
print(df_baseline[["true_accuracy", "pseudo_reward"]].describe().round(3))


In [ ]:
# Visualise the gap between pseudo-reward and true accuracy — this IS the reward hack signal
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Baseline SRT: Pseudo-Reward vs True Accuracy on 100 DAPO Samples", fontsize=13)

x = range(len(df_baseline))
axes[0].bar(x, df_baseline["true_accuracy"],  color="steelblue", alpha=0.7, label="True accuracy")
axes[0].bar(x, df_baseline["pseudo_reward"],  color="orange",    alpha=0.4, label="Pseudo-reward")
axes[0].set_xlabel("Prompt index")
axes[0].set_ylabel("Score")
axes[0].set_title("Per-prompt: True Accuracy vs Pseudo-Reward")
axes[0].legend()

axes[1].scatter(df_baseline["pseudo_reward"], df_baseline["true_accuracy"], alpha=0.5)
axes[1].plot([0, 1], [0, 1], "r--", label="y=x (ideal)")
axes[1].set_xlabel("Pseudo-Reward (self-consistency)")
axes[1].set_ylabel("True Accuracy")
axes[1].set_title("Reward Hack: Gap between pseudo-reward and accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("baseline_reward_gap.png", dpi=150)
plt.show()
print("Saved: baseline_reward_gap.png")

---
## Step 2 — Code the Custom Entropy-Augmented Reward Function

$$R_{total} = R_{consistency} + \alpha \, \mathcal{H}_{batch}$$

This extends `SelfLearningRewardManager` from `verl/workers/reward_manager/self_learning.py`.
All scoring uses the same `_default_compute_score` / `_extract_verifiable_part_of_solution` calls.

In [ ]:
def compute_batch_entropy(
    answers: List[Optional[str]],
    n_generations: int,
) -> float:
    """
    Normalised Shannon entropy of the answer distribution for one prompt's rollout batch.

    Steps (matching the paper derivation):
      1. Extract unique answers and count occurrences.
      2. Compute P(a) = count(a) / total_parseable.
      3. H = -sum_a P(a) log P(a).
      4. Normalise by log(n_generations) so H ∈ [0, 1].

    Returns 0.0 when all generations are unparseable (no \\boxed{} found).
    """
    valid = [a for a in answers if a is not None]
    if not valid:
        return 0.0

    counts  = Counter(valid)
    total   = len(valid)
    entropy = -sum((c / total) * math.log(c / total + 1e-12) for c in counts.values())

    max_entropy = math.log(n_generations)
    return float(np.clip(entropy / max_entropy, 0.0, 1.0)) if max_entropy > 0 else 0.0


# --- Sanity checks ---
assert compute_batch_entropy(["42"] * 16, 16) < 0.01,  "collapsed → H ≈ 0"
assert compute_batch_entropy([str(i) for i in range(16)], 16) > 0.99, "fully diverse → H ≈ 1"
print("compute_batch_entropy: OK")

In [ ]:
class EntropyAugmentedRewardManager:
    """
    Extends SelfLearningRewardManager (verl/workers/reward_manager/self_learning.py)
    with a per-batch entropy bonus:

        R_total = R_consistency + alpha * H_batch

    Scoring uses the identical verl helpers (_default_compute_score,
    _extract_verifiable_part_of_solution) so results are directly comparable.
    """

    def __init__(
        self,
        alpha: float,
        n_generations: int = N_GENERATIONS,
        self_consistency_threshold: float = 0.0,
        soft_reward: bool = False,
        data_source: str = DATA_SOURCE,
    ):
        self.alpha = alpha
        self.n_generations = n_generations
        self.self_consistency_threshold = self_consistency_threshold
        self.soft_reward = soft_reward
        self.data_source = data_source

    # ------------------------------------------------------------------
    # Step 1: parse all \boxed{} answers from the batch
    # ------------------------------------------------------------------
    def parse_answers(self, responses: List[str]) -> List[Optional[str]]:
        return [extract_answer(r, self.data_source) for r in responses]

    # ------------------------------------------------------------------
    # Step 2+3: P(a) and H_batch
    # ------------------------------------------------------------------
    def batch_entropy(self, extracted: List[Optional[str]]) -> float:
        return compute_batch_entropy(extracted, self.n_generations)

    # ------------------------------------------------------------------
    # Majority-vote self-labeling (identical to SelfLearningRewardManager)
    # ------------------------------------------------------------------
    def majority_vote(self, extracted: List[Optional[str]]) -> Optional[str]:
        valid = [a for a in extracted if a is not None]
        if not valid:
            return None
        most_common, freq = Counter(valid).most_common(1)[0]
        if freq / len(valid) >= self.self_consistency_threshold:
            return most_common
        return None

    # ------------------------------------------------------------------
    # Step 4: R_total = R_consistency + alpha * H_batch
    # ------------------------------------------------------------------
    def compute_rewards(
        self,
        responses: List[str],
        ground_truth: str,
    ) -> Dict:
        """
        Args:
            responses:    list of raw LLM strings for one prompt
            ground_truth: known answer (or SELF_LABEL_SENTINEL for unlabeled)
        """
        extracted = self.parse_answers(responses)

        # --- self-label if needed (mirrors SelfLearningRewardManager) ---
        if ground_truth == SELF_LABEL_SENTINEL:
            reference = self.majority_vote(extracted)
        else:
            reference = ground_truth

        # --- R_consistency per generation ---
        if reference is None:
            consistency_rewards = [0.0] * len(responses)
        elif self.soft_reward and ground_truth == SELF_LABEL_SENTINEL:
            counts = Counter(a for a in extracted if a is not None)
            total  = len([a for a in extracted if a is not None]) or 1
            consistency_rewards = [
                counts.get(a, 0) / total if a is not None else 0.0
                for a in extracted
            ]
        else:
            consistency_rewards = [
                compute_score(r, reference, self.data_source) for r in responses
            ]

        # --- H_batch (same for all rollouts of this prompt) ---
        H             = self.batch_entropy(extracted)
        entropy_bonus = self.alpha * H

        # --- R_total ---
        total_rewards = [r_c + entropy_bonus for r_c in consistency_rewards]

        return {
            "consistency_rewards": consistency_rewards,
            "entropy":             H,
            "entropy_bonus":       entropy_bonus,
            "total_rewards":       total_rewards,
            "extracted_answers":   extracted,
            "self_label":          reference,
        }


# Quick demo with the actual rollouts we already collected
demo_manager = EntropyAugmentedRewardManager(alpha=0.1)
demo_result  = demo_manager.compute_rewards(
    responses=baseline_rollouts[0]["responses"],
    ground_truth=SELF_LABEL_SENTINEL,
)
print("Prompt 0 demo:")
print(f"  Hidden answer   : {test_set[0]['hidden_answer']}")
print(f"  Self-label      : {demo_result['self_label']}")
print(f"  Batch entropy H : {demo_result['entropy']:.4f}")
print(f"  Entropy bonus   : {demo_result['entropy_bonus']:.4f}  (α=0.1)")
print(f"  Consistency rews: {demo_result['consistency_rewards']}")
print(f"  Total rewards   : {[round(x,4) for x in demo_result['total_rewards']]}")

### 2b — verl Integration Patch

The snippet below shows **exactly where** to add the entropy bonus inside
`verl/workers/reward_manager/self_learning.py → __call__`.

In [ ]:
# Reference only — shows the two-line addition inside SelfLearningRewardManager.__call__
# after reward_tensor[i, valid_response_length - 1] = reward

PATCH = """
# ── PATCH: add alpha to __init__ ─────────────────────────────────────
# self.alpha = alpha  (new param, default 0.0 for backwards compat)

# ── PATCH: after the main per-generation loop, add this block ────────
if self.alpha > 0.0:
    # Group extracted answers by prompt
    prompt_to_extracted = defaultdict(list)
    for p, ex in zip(prompt_list, extracted_answer_list):
        prompt_to_extracted[p].append(ex)

    # Compute H per prompt
    prompt_to_H = {
        p: compute_batch_entropy(answers, n_generations=self.n_generations)
        for p, answers in prompt_to_extracted.items()
    }

    # Add entropy bonus at the last response token (same position as consistency reward)
    for i, p in enumerate(prompt_list):
        valid_len = data[i].batch['attention_mask'][prompt_length:].sum()
        reward_tensor[i, valid_len - 1] += self.alpha * prompt_to_H[p]
# ─────────────────────────────────────────────────────────────────────
"""
print(PATCH)

---
## Step 3 — Small-Scale α Tuning on the 100 Samples

We reuse the **same rollouts** generated in Step 1 (no extra inference needed) and score them
with three different α values.

In [ ]:
ALPHA_RUNS = {
    "Run A  α=0.01 (weak)":     0.01,
    "Run B  α=0.10 (moderate)": 0.10,
    "Run C  α=0.50 (strict)":   0.50,
}

alpha_results = {}  # label → list of per-prompt dicts

for label, alpha in ALPHA_RUNS.items():
    manager = EntropyAugmentedRewardManager(alpha=alpha)
    run_rows = []

    for row, base in zip(test_set, baseline_rollouts):
        result = manager.compute_rewards(
            responses=base["responses"],
            ground_truth=SELF_LABEL_SENTINEL,
        )

        # True accuracy uses the hidden ground truth (never given to the model)
        true_acc = np.mean([
            compute_score(r, row["hidden_answer"]) for r in base["responses"]
        ])

        run_rows.append({
            "true_accuracy":      true_acc,
            "pseudo_reward":      result["consistency_rewards"][0],  # consistency part
            "entropy":            result["entropy"],
            "total_reward_mean":  float(np.mean(result["total_rewards"])),
            "entropy_bonus":      result["entropy_bonus"],
        })

    alpha_results[label] = run_rows

    df_run = pd.DataFrame(run_rows)
    print(f"{label:35s} → acc={df_run['true_accuracy'].mean():.3f}  "
          f"pseudo_r={df_run['pseudo_reward'].mean():.3f}  "
          f"entropy={df_run['entropy'].mean():.3f}  "
          f"total_r={df_run['total_reward_mean'].mean():.3f}")

In [ ]:
metrics_to_plot = [
    ("true_accuracy",     "True Accuracy"),
    ("pseudo_reward",     "Pseudo-Reward (consistency)"),
    ("entropy",           "Batch Entropy (normalised)"),
    ("total_reward_mean", "Total Reward (R_c + α H)"),
]

COLORS = ["steelblue", "orange", "green", "red"]
all_runs = {"Baseline  α=0": baseline_rollouts, **{k: None for k in ALPHA_RUNS}}

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("α Tuning: Entropy Bonus Effect on 100-Sample DAPO Test Set", fontsize=13, fontweight="bold")
axes = axes.flatten()

# Build per-run metric arrays
def extract_metric(rows, metric):
    if metric == "true_accuracy":
        return [r["true_accuracy"] for r in rows] if "true_accuracy" in rows[0] else [r["true_accuracy"] for r in rows]
    return [r.get(metric, r.get("true_accuracy")) for r in rows]

baseline_rows = [{
    "true_accuracy": r["true_accuracy"],
    "pseudo_reward": r["pseudo_reward"],
    "entropy": compute_batch_entropy(r["extracted"], N_GENERATIONS),
    "total_reward_mean": r["pseudo_reward"],  # no entropy bonus
} for r in baseline_rollouts]

run_data = {"Baseline  α=0": baseline_rows}
for label in ALPHA_RUNS:
    run_data[label] = alpha_results[label]

for ax_idx, (metric, title) in enumerate(metrics_to_plot):
    ax = axes[ax_idx]
    for color, (label, rows) in zip(COLORS, run_data.items()):
        vals = [r[metric] for r in rows]
        ax.plot(vals, color=color, linewidth=1.2,
                linestyle="--" if "Baseline" in label else "-",
                label=label, alpha=0.85)
        # also plot smoothed mean
        ax.axhline(np.mean(vals), color=color, linewidth=2, linestyle=":", alpha=0.6)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Prompt index")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("alpha_tuning_100samples.png", dpi=150)
plt.show()
print("Saved: alpha_tuning_100samples.png")

### 3b — Diversity sanity check

Verify that high entropy in a batch comes from genuine math diversity, not from the model spamming random tokens.

In [ ]:
_MATH_RE = re.compile(r"^-?[0-9]+([./][0-9]+)?(\s*[\+\-\*/]\s*-?[0-9]+)*$")

def audit_diversity(base_rollouts: List[Dict], top_n_highest_entropy: int = 5):
    """
    For the top-n highest-entropy prompts, print the extracted answers
    and flag if entropy looks like random noise rather than genuine math diversity.
    """
    rows = [
        {
            "prompt_snip": r["prompt"][:80],
            "extracted":   r["extracted"],
            "entropy":     compute_batch_entropy(r["extracted"], N_GENERATIONS),
        }
        for r in base_rollouts
    ]
    rows.sort(key=lambda x: x["entropy"], reverse=True)

    print(f"Top {top_n_highest_entropy} highest-entropy prompts:")
    for i, row in enumerate(rows[:top_n_highest_entropy]):
        valid  = [a for a in row["extracted"] if a is not None]
        parsed = len(valid) / len(row["extracted"]) if row["extracted"] else 0
        math_v = sum(_MATH_RE.match(a or "") is not None for a in valid) / max(len(valid), 1)
        counts = Counter(valid).most_common(4)

        print(f"\n  [{i+1}] H={row['entropy']:.3f}  parseable={parsed:.0%}  math-valid={math_v:.0%}")
        print(f"       Prompt: {row['prompt_snip']}...")
        print(f"       Top answers: {counts}")

        if row["entropy"] > 0.5 and math_v < 0.5:
            print("       ⚠  HIGH ENTROPY + LOW MATH VALIDITY → possible gaming")
        elif row["entropy"] > 0.5:
            print("       ✓  High entropy with valid math answers → genuine diversity")
        else:
            print("       ✓  Moderate/low entropy → healthy")

audit_diversity(baseline_rollouts)

---
## Step 4 — Analyze the Stabilized Metrics

Compare best-α run vs baseline. Check:
1. **KL proxy** — diversity stays high (low KL to base distribution).  
2. **Accuracy** — stays stable or improves.

In [ ]:
def summarize(label: str, rows: List[Dict]) -> Dict:
    df = pd.DataFrame(rows)
    return {
        "label":           label,
        "mean_accuracy":   df["true_accuracy"].mean(),
        "std_accuracy":    df["true_accuracy"].std(),
        "mean_pseudo_r":   df["pseudo_reward"].mean(),
        "mean_entropy":    df["entropy"].mean(),
        "mean_total_r":    df["total_reward_mean"].mean(),
        # approximate KL proxy: lower entropy → model collapsed → higher implicit KL from ref
        "kl_proxy":        1.0 - df["entropy"].mean(),
    }

summary_rows = [summarize("Baseline α=0", baseline_rows)]
for label, rows in alpha_results.items():
    summary_rows.append(summarize(label, rows))

df_summary = pd.DataFrame(summary_rows).set_index("label").round(4)
print(df_summary.to_string())

# Pick best alpha: maximise accuracy, minimise kl_proxy
df_alpha_only = df_summary.drop("Baseline α=0")
df_alpha_only["score"] = df_alpha_only["mean_accuracy"] - df_alpha_only["kl_proxy"]
best_label = df_alpha_only["score"].idxmax()
best_alpha = ALPHA_RUNS[best_label]
print(f"\n→ Best α: {best_alpha}  ({best_label})")

In [ ]:
# Side-by-side bar chart: all runs
labels   = list(df_summary.index)
acc_vals = df_summary["mean_accuracy"].values
pr_vals  = df_summary["mean_pseudo_r"].values
ent_vals = df_summary["mean_entropy"].values
kl_vals  = df_summary["kl_proxy"].values

x   = np.arange(len(labels))
w   = 0.2
fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(x - 1.5*w, acc_vals, w, label="True Accuracy",        color="steelblue")
ax.bar(x - 0.5*w, pr_vals,  w, label="Pseudo-Reward",        color="orange")
ax.bar(x + 0.5*w, ent_vals, w, label="Batch Entropy",        color="green")
ax.bar(x + 1.5*w, kl_vals,  w, label="KL Proxy (1-entropy)", color="red", alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.set_title("Summary: Baseline vs α Runs (100 DAPO samples)", fontsize=13)
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("metric_summary_bar.png", dpi=150)
plt.show()
print("Saved: metric_summary_bar.png")

---
## Step 5 — Scale Up to the Full Experiment

### 5a — Curate the easy one-third subset using real model pass rates

In [ ]:
# We already have pass rates for our 100-sample test set from Step 1.
# For the full dataset, run the same generate_rollouts loop over all prompts
# (resource-intensive — run on GPU server, not this notebook cell).
#
# Here we compute pass rates for our 100 samples to demo the selection logic.

pass_rate_rows = [
    {
        "original_idx": test_set[i]["original_idx"],
        "prompt":       test_set[i]["prompt"],
        "pass_rate":    baseline_rollouts[i]["true_accuracy"],
    }
    for i in range(len(test_set))
]
df_pass = pd.DataFrame(pass_rate_rows)

# Easy one-third: prompts where pass_rate ≤ 33rd percentile
# (same definition as in the SRT paper — challenging enough to train on)
cutoff      = df_pass["pass_rate"].quantile(1 / 3)
easy_subset = df_pass[df_pass["pass_rate"] <= cutoff]

print(f"Pass-rate cutoff (33rd pctile): {cutoff:.3f}")
print(f"Easy subset size: {len(easy_subset)} / {len(df_pass)}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_pass["pass_rate"], bins=20, color="steelblue", alpha=0.7)
ax.axvline(cutoff, color="red", linewidth=2, label=f"Easy cutoff ({cutoff:.2f})")
ax.set_xlabel("Pass Rate (true accuracy over 16 rollouts)")
ax.set_ylabel("# Prompts")
ax.set_title("DAPO Pass Rate Distribution — 100-Sample Subset")
ax.legend()
plt.tight_layout()
plt.savefig("pass_rate_distribution.png", dpi=150)
plt.show()

### 5b — Multi-model comparison setup

Run Steps 1–4 for each model by changing `MODEL_ID` at the top and re-running.
This cell collects final summary numbers assuming you've saved them to a CSV.

In [ ]:
ALL_MODELS = [
    "Qwen/Qwen2.5-Math-7B",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "AI-MO/NuminaMath-7B-CoT",
]

# To run all three models, either:
#   1. Loop below and reload the model each time (sequential, slow), OR
#   2. Run the notebook once per model and save results to CSV.
#
# Single-model run (current MODEL_ID already evaluated above):
current_summary = summarize(MODEL_ID, alpha_results.get(best_label, baseline_rows))
current_summary["model"] = MODEL_ID
current_summary["alpha"] = best_alpha
print("Current model summary:")
print(pd.Series(current_summary))

# Optionally save to CSV so you can merge results from multiple runs
results_csv = "multi_model_results.csv"
df_current  = pd.DataFrame([current_summary])

if os.path.exists(results_csv):
    df_prev = pd.read_csv(results_csv)
    df_all  = pd.concat([df_prev, df_current], ignore_index=True).drop_duplicates(subset=["model"])
else:
    df_all = df_current

df_all.to_csv(results_csv, index=False)
print(f"\nSaved to {results_csv}")
print(df_all[["model", "alpha", "mean_accuracy", "mean_entropy", "kl_proxy"]])

### 5c — Launch full verl training with tuned α

Once the 100-sample test confirms the math, run the real RLOO training job:

In [ ]:
import subprocess

# Writes a launch script that inherits everything from srt.sh and adds ALPHA
launch_script = f"""#!/bin/bash
# Auto-generated from entropy_reward_experiment.ipynb
# Tuned alpha: {best_alpha}

source experiment_scripts/srt.sh

# Entropy bonus hyperparameter (new)
export ENTROPY_ALPHA={best_alpha}

python3 -m verl.trainer.main_ppo \\\n"""

# Show what the launch looks like — don't actually run it here
print(launch_script[:800] + "\n... (full args inherited from srt.sh)")

with open("run_entropy_srt.sh", "w") as f:
    f.write(launch_script)

print("\nWrote run_entropy_srt.sh — edit TRAIN_DATASET_PATH and MODEL_PATH, then run on GPU cluster.")

---
## Summary

| Step | What we did | Key artifact |
|---|---|---|
| 1 | Loaded DAPO + the real model via `transformers`; generated 16 rollouts per prompt for 100 samples; scored with `verl.utils.reward_score` | `baseline_reward_gap.png` |
| 2 | Implemented `EntropyAugmentedRewardManager` (same scoring functions as `SelfLearningRewardManager`) | `EntropyAugmentedRewardManager` class |
| 3 | Swept α ∈ {0.01, 0.10, 0.50} on the same 100 rollouts | `alpha_tuning_100samples.png` |
| 4 | Chose best α by highest accuracy + lowest KL proxy | `metric_summary_bar.png` |
| 5 | Curated easy-⅓ subset; exported launch script with tuned α | `run_entropy_srt.sh` |

**To reproduce on GPU cluster:**
1. Apply the two-line patch from Step 2b to `verl/workers/reward_manager/self_learning.py`.
2. Set `TRAIN_DATASET_PATH` and `MODEL_PATH` in `run_entropy_srt.sh`.
3. Run `bash run_entropy_srt.sh` and monitor WandB for accuracy, pseudo-reward, and KL.